# Running the Best Model — Voting (Random Forest + Bagging), 6-class habitat guild

This loads the final model saved by `05_six_class_ceiling_analysis.ipynb` (`models/final_model_v6_6class.pkl`, macro-F1 0.779, accuracy 74.8%, the best result across v3-v6) and runs inference. No retraining — just load and predict, the way guideline item #15 ("accept user input and perform inference") expects.

The saved model is a plain `sklearn.ensemble.VotingClassifier` over `RandomForestClassifier` + `BaggingClassifier` — no custom classes, so it unpickles with just scikit-learn installed, in any notebook or script.


## 1. Load the saved model, scaler, and feature schema

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd

MODELS_DIR = PROJECT_ROOT / "models"

model = joblib.load(MODELS_DIR / "final_model_v6_6class.pkl")
scaler = joblib.load(MODELS_DIR / "final_scaler_v6.pkl")          # not needed by this model (tree-based), kept for completeness / other saved models
feature_columns = joblib.load(MODELS_DIR / "final_feature_columns_v6.pkl")

print("Model:", model)
print("\nClasses:", list(model.classes_))
print("\nExpects", len(feature_columns), "feature columns (after one-hot encoding)")


Model: VotingClassifier(estimators=[('Random Forest',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     max_features=0.5,
                                                     n_estimators=200,
                                                     n_jobs=-1,
                                                     random_state=42)),
                             ('Bagging',
                              BaggingClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                                                 max_depth=15,
                                                                                 random_state=42),
                                                max_features=0.5,
                                                max_samples=1.0,
                                                n_e

## 2. Accuracy on the real held-out test set

This reproduces the exact same train/test split used during training (`random_state=42`, stratified 80/20 -- identical to `05_six_class_ceiling_analysis.ipynb`), so the test rows here were never seen during training, and evaluates the loaded model against it. This is the honest, reusable way to check the model's real accuracy from this notebook, rather than eyeballing a handful of sample predictions.

In [2]:
NUMERIC_COLS = [
    "decimalLatitude", "decimalLongitude", "avg_rad", "NDVI", "elevation_meters",
    "Carbon_Mass", "Dust_Mass", "SO2_Mass", "Sulfate_Mass", "Sea_Salt_Mass",
    "Total_Aerosol_Extinction", "temp_mean", "rainfall", "wind_mean",
    "humid_mean", "shortwave_radiation", "lka_general_2020",
]
CATEGORICAL_COLS = ["stateProvince", "LandCover_Class"]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "bird_habitat_guild_6class_classification.csv"
df = pd.read_csv(DATA_PATH)

X_full = pd.get_dummies(df[NUMERIC_COLS + CATEGORICAL_COLS], columns=CATEGORICAL_COLS, drop_first=True)
y_full = df["habitat_guild_6"]

_, X_test, _, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=42
)
X_test = X_test.reindex(columns=feature_columns, fill_value=0)

preds = model.predict(X_test)
print(f"Accuracy:          {accuracy_score(y_test, preds):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Macro-F1:          {f1_score(y_test, preds, average='macro'):.4f}")
print()
print(classification_report(y_test, preds))

Accuracy:          0.7481
Balanced accuracy: 0.7964
Macro-F1:          0.7792

                                     precision    recall  f1-score   support

            Coastal (tern + curlew)       0.70      0.67      0.68       463
   Harpactes fasciatus (rainforest)       0.91      0.95      0.93       350
        Hirundo domicola (highland)       0.82      0.89      0.85       255
  Lowland open-country/wetland-edge       0.67      0.66      0.67       717
Motacilla alba (urban/wetland-edge)       0.76      0.92      0.83       128
         Upland scrub/woodland-edge       0.74      0.69      0.71       489

                           accuracy                           0.75      2402
                          macro avg       0.77      0.80      0.78      2402
                       weighted avg       0.75      0.75      0.75      2402



## 3. Build the exact input row the model expects

The model was trained on one-hot encoded `stateProvince` + `LandCover_Class` (`pd.get_dummies(..., drop_first=True)`) alongside the raw numeric columns. A new raw observation has to be converted the same way and reindexed onto the **exact same 51 columns, in the same order** — any dummy column missing from the new observation (e.g. a province that isn't the one just entered) must be filled with 0, and any dummy column in the new data that the model never saw during training must be dropped. `reindex(columns=feature_columns, fill_value=0)` does both in one call.


In [4]:
NUMERIC_COLS = [
    "decimalLatitude", "decimalLongitude", "avg_rad", "NDVI", "elevation_meters",
    "Carbon_Mass", "Dust_Mass", "SO2_Mass", "Sulfate_Mass", "Sea_Salt_Mass",
    "Total_Aerosol_Extinction", "temp_mean", "rainfall", "wind_mean",
    "humid_mean", "shortwave_radiation", "lka_general_2020",
]
CATEGORICAL_COLS = ["stateProvince", "LandCover_Class"]

def prepare_input(raw_rows: pd.DataFrame) -> pd.DataFrame:
    '''raw_rows: a DataFrame with the original (un-encoded) columns -- same
    shape as one row of data/processed/bird_habitat_guild_6class_classification.csv,
    minus the target. Returns a DataFrame aligned to the model's expected columns.'''
    encoded = pd.get_dummies(raw_rows[NUMERIC_COLS + CATEGORICAL_COLS], columns=CATEGORICAL_COLS)
    return encoded.reindex(columns=feature_columns, fill_value=0)


## 4. Inference on real held-out rows (sanity check against the notebook's own test results)

In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "bird_habitat_guild_6class_classification.csv"
df = pd.read_csv(DATA_PATH)

sample_raw = df.sample(n=5, random_state=7)
X_sample = prepare_input(sample_raw)

preds = model.predict(X_sample)
probas = model.predict_proba(X_sample)

results = pd.DataFrame({
    "true_guild": sample_raw["habitat_guild_6"].values,
    "predicted_guild": preds,
    "confidence": probas.max(axis=1).round(3),
})
results


,true_guild,predicted_guild,confidence
0,Lowland open-country/wetland-edge,Lowland open-country/wetland-edge,0.722
1,Lowland open-country/wetland-edge,Lowland open-country/wetland-edge,0.498
2,Lowland open-country/wetland-edge,Lowland open-country/wetland-edge,0.640
3,Lowland open-country/wetland-edge,Lowland open-country/wetland-edge,0.618
4,Harpactes fasciatus (rainforest),Harpactes fasciatus (rainforest),0.994


## 5. Inference on a single hand-entered observation (the inference use case guideline #15 asks for)

In [6]:
new_observation = pd.DataFrame([{
    "decimalLatitude": 6.93,
    "decimalLongitude": 79.85,
    "avg_rad": 2.1,
    "NDVI": 0.58,
    "elevation_meters": 15,
    "Carbon_Mass": 0.5,
    "Dust_Mass": 1.2,
    "SO2_Mass": 0.3,
    "Sulfate_Mass": 0.8,
    "Sea_Salt_Mass": 1.5,
    "Total_Aerosol_Extinction": 0.15,
    "temp_mean": 27.5,
    "rainfall": 4.8,
    "wind_mean": 3.9,
    "humid_mean": 81.0,
    "shortwave_radiation": 210.0,
    "lka_general_2020": 0.4,
    "stateProvince": "Gampaha",
    "LandCover_Class": 11,
}])

X_new = prepare_input(new_observation)
pred_guild = model.predict(X_new)[0]
pred_probs = pd.Series(model.predict_proba(X_new)[0], index=model.classes_).sort_values(ascending=False)

print("Predicted habitat guild:", pred_guild)
print("\nClass probabilities:")
print(pred_probs.round(4))


Predicted habitat guild: Coastal (tern + curlew)

Class probabilities:
Coastal (tern + curlew)                0.3484
Upland scrub/woodland-edge             0.1950
Harpactes fasciatus (rainforest)       0.1856
Lowland open-country/wetland-edge      0.1764
Motacilla alba (urban/wetland-edge)    0.0817
Hirundo domicola (highland)            0.0129
dtype: float64


## 6. Edit and re-run

Change any value in the `new_observation` dict above (e.g. push `elevation_meters` up to 1500 and `LandCover_Class`/`NDVI` to forest-like values) and re-run cells 4-5 to see the prediction change. This is exactly the loop a front-end UI (guideline #14/#15) would wrap in text boxes — a form posting these same field names to a backend that calls `prepare_input()` + `model.predict_proba()` is the whole app.
